# Huấn luyện Mô hình Nhận diện Cử chỉ tay (Smart Hospital)
**LƯU Ý QUAN TRỌNG:** Trên thanh Menu, hãy vào `Runtime` (Thời gian chạy) -> `Change runtime type` (Thay đổi loại thời gian chạy) -> Chọn **T4 GPU** -> Lưu.

**Sau đó, chỉ cần vào `Runtime` -> `Run all` (Chạy tất cả) và đợi kết quả!**

In [ ]:
# 1. KẾT NỐI GOOGLE DRIVE ĐỂ LẤY DATASET
from google.colab import drive
import os
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/SmartHospital/dataset_3class_new.zip'
if not os.path.exists(ZIP_PATH):
    ZIP_PATH = '/content/drive/MyDrive/SmartHospital/dataset.zip'
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError("❌ LỖI: Không tìm thấy file dataset_3class_new.zip hay dataset.zip trong MyDrive/SmartHospital/. Vui lòng upload lên Drive trước!")
print('✅ Đã tìm thấy dataset:', ZIP_PATH)

In [ ]:
# 2. GIẢI NÉN DATASET CHUẨN
import shutil
!rm -rf /content/dataset
!mkdir -p /content/dataset
!unzip -q {ZIP_PATH} -d /content/dataset/
print('✅ Đã giải nén xong! Kiểm tra file data.yaml:')
!cat /content/dataset/data.yaml

In [ ]:
# 3. CÀI ĐẶT YOLOV8 VÀ KIỂM TRA GPU
!pip install ultralytics -q
import torch
print('\n✅ CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✅ GPU đang dùng:', torch.cuda.get_device_name(0))
else:
    print('❌ CẢNH BÁO: Chưa bật GPU. Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU rồi chạy lại!')

In [ ]:
# 4. BẮT ĐẦU HUẤN LUYỆN (TRAIN)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='gesture_hospital_v1',
    exist_ok=True,  # Tránh lỗi tự động sinh thư mục mới (v1-2, v1-3...)
    patience=20
)

In [ ]:
# 5. KIỂM TRA LẠI ĐỘ CHÍNH XÁC
from ultralytics import YOLO
best_path = '/content/runs/gesture_hospital_v1/weights/best.pt'
model = YOLO(best_path)
metrics = model.val(data='/content/dataset/data.yaml')
print("\n🎯 Độ chính xác mAP50-95:", metrics.box.map)

In [ ]:
# 6. COPY MODEL BEST.PT VỀ LẠI GOOGLE DRIVE
!cp /content/runs/gesture_hospital_v1/weights/best.pt "/content/drive/MyDrive/SmartHospital/best.pt"
print('🎉 XONG! Đã lưu thành công file best.pt vào thư mục SmartHospital trên Google Drive của bạn!')